# First Machine Learning Classifier

### Titanic Passenger Survival Prediction

**Project:** First Machine Learning Classifier  
**Model:** Decision Tree Classifier  
**Library:** Scikit-learn

## What I am trying to find out

I want to check whether a passenger's ticket class, sex, age and fare can help predict if they survived the Titanic disaster.

The target column is `survived`. The other four columns are the inputs used by the model.

This is a small learning project. The model can find patterns in this dataset, but it cannot explain every person's actual experience.


## 1. Import the libraries

I use pandas to handle the data, matplotlib and seaborn for charts, and Scikit-learn to train and check the model.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    mean_squared_error,
    classification_report
)


## 2. Load and have a look at the data

I am using the Titanic dataset available through seaborn. First, I check a few rows and the missing values before preparing it.


In [ ]:
titanic = sns.load_dataset("titanic")

data = titanic[["survived", "pclass", "sex", "age", "fare"]].copy()

print("First five rows:")
display(data.head())

print("\\nMissing values:")
print(data.isnull().sum())

print("\\nDataset size before cleaning:", data.shape)


## 3. Prepare the columns

For this first model, I remove rows where one of the selected columns is missing. I convert `sex` into a number because the model needs numeric input: female is 1 and male is 0.


In [ ]:
data = data.dropna().copy()
data["sex"] = (data["sex"] == "female").astype(int)

print("Dataset size after cleaning:", data.shape)
display(data.head())


## 4. A quick look at survival

This chart gives a simple view of how many passengers in this cleaned sample survived and how many did not. It is only an overview, not a model result.


In [ ]:
survival_counts = data["survived"].value_counts().sort_index()

survival_counts.plot(kind="bar", rot=0)
plt.xticks([0, 1], ["Did not survive", "Survived"])
plt.ylabel("Number of passengers")
plt.title("Survival count in the selected data")
plt.tight_layout()
plt.show()


## 5. Separate the inputs and target

`X` contains the four input columns. `y` contains the answer the model is learning to predict.


In [ ]:
X = data[["pclass", "sex", "age", "fare"]]
y = data["survived"]

print("Input columns:", list(X.columns))
print("Target column: survived")


## 6. Split into training and test data

I keep 20% of the rows for testing. The model will not train on these rows. `stratify=y` helps keep a similar survival ratio in both parts.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 7. Compare with a simple baseline

Before using a decision tree, I check how well a very basic guess would do. The baseline always predicts the most common class in the training data.

A model should do better than this simple starting point to be useful.


In [ ]:
majority_class = y_train.mode()[0]
baseline_predictions = [majority_class] * len(y_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print("Most common training outcome:", majority_class)
print("Baseline accuracy:", round(baseline_accuracy, 3))


## 8. Train the decision tree

I set `max_depth=4` to keep the tree from becoming unnecessarily complicated. The model is fitted using the training data only.


In [ ]:
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)


## 9. Check the model

Accuracy tells me the fraction of test passengers predicted correctly. The confusion matrix gives more detail about correct and incorrect predictions.

For this matrix, rows are the actual values and columns are the predicted values:
- Top-left: actually did not survive, predicted did not survive.
- Top-right: actually did not survive, predicted survived.
- Bottom-left: actually survived, predicted did not survive.
- Bottom-right: actually survived, predicted survived.


In [ ]:
model_accuracy = accuracy_score(y_test, predictions)
matrix = confusion_matrix(y_test, predictions)

# MSE is included because it is listed in the project requirements.
# For this binary classification task, it is the average squared difference
# between the actual 0/1 labels and predicted 0/1 labels.
classification_mse = mean_squared_error(y_test, predictions)

print("Baseline accuracy:", round(baseline_accuracy, 3))
print("Decision tree accuracy:", round(model_accuracy, 3))
print("Classification MSE:", round(classification_mse, 3))
print("\\nConfusion matrix:")
print(matrix)

print("\\nClassification report:")
print(classification_report(y_test, predictions, target_names=["Did not survive", "Survived"]))


In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted no", "Predicted yes"],
    yticklabels=["Actual no", "Actual yes"]
)
plt.title("Confusion Matrix")
plt.xlabel("Prediction")
plt.ylabel("Actual result")
plt.tight_layout()
plt.show()


## 10. Check whether tree depth changes the result

A very deep tree can memorise details from the training data. I try depths from 1 to 12 and compare training accuracy with test accuracy.

The test score is the more important one when thinking about new passengers.


In [ ]:
depth_values = range(1, 13)
training_scores = []
testing_scores = []

for depth in depth_values:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)

    training_scores.append(tree.score(X_train, y_train))
    testing_scores.append(tree.score(X_test, y_test))

plt.figure(figsize=(8, 4))
plt.plot(list(depth_values), training_scores, marker="o", label="Training accuracy")
plt.plot(list(depth_values), testing_scores, marker="o", label="Test accuracy")
plt.xlabel("Maximum tree depth")
plt.ylabel("Accuracy")
plt.title("Decision tree depth comparison")
plt.xticks(list(depth_values))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Which inputs did the tree use most?

Feature importance shows how much each input helped the trained tree make splits. It is useful for a quick comparison, but it does not prove that a feature caused someone to survive.


In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance.round(3))

importance.sort_values().plot(kind="barh")
plt.xlabel("Feature importance")
plt.title("Inputs used by the decision tree")
plt.tight_layout()
plt.show()


## 12. My project observations

Run the notebook from top to bottom and use the values printed above to complete these notes. I have left the result spaces blank so the report contains the actual output from your run, rather than made-up numbers.

- Baseline accuracy: ______
- Decision tree test accuracy: ______
- Classification MSE: ______
- Most important input according to this tree: ______
- Best test accuracy among the depths I tried: ______

### What I learned

I learned how to separate input columns from the target, split data into training and test sets, train a Scikit-learn decision tree and evaluate predictions on data the model did not train on. Comparing the model with the baseline helped me check whether the model was doing better than a simple majority guess.

### Limitation

This model only uses ticket class, sex, age and fare. It leaves out other details, and rows with missing values were removed. The dataset is also from one historical event, so its patterns should not be treated as a general rule for other situations.
